In [1]:
import os
import math
import time
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter
from problems.ThickenShell import ThickenShell
from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADDomainVisualizer import CADDomainVisualizer
from Utils.CADVisualizer   import CADVisualizer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell
from Training.MainTrain import NN_Trainer, TrainingConfig
from Decoder_CLasses.ContinuousVoronoiDecoder import ContinuousVoronoiDecoder
from Decoder_CLasses.NearUniformHoneycombBaseline import NearUniformHoneycombBaseline
from Decoder_CLasses.IntrinsicSurfaceHoneycombBaseline import IntrinsicSurfaceHoneycombBaseline
from Pred_NN_Classes.ppnet import PPNet
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d, cKDTree,QhullError
from Utils.ExportAbaqus_VoxelBased import export_abaqus_voxel_fem
from Utils.ExportAbaqus import export_abaqus_shell
from Utils.embedded_centroid_stress_audit import (
    export_embedded_centroid_stress_audit,
    print_stress_case_summary,
)
from Utils.RunFEM_ExportVoxelABA import VoxelFEMAbaqusRunner
from Decoder_CLasses import Phase1VoronoiDecoder

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)

%matplotlib inline
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

Code Directory: /home/arash/HVD_SeedsBase
Test Step files Directory: /home/arash/HVD_SeedsBase/Testparts
device: cuda
PyVista backend: trame


In [2]:

def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def show_surface_density(fields, density_key="rho"):
    face_tensor = fields["face_tensor"]

    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary.
    if triangles.min() == 1:
        triangles = triangles - 1

    # PyVista face format:
    # [3, i, j, k, 3, i, j, k, ...]
    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    mesh = pv.PolyData(points, pv_faces)

    density = to_numpy(
        fields[density_key]
    ).reshape(-1)

    mesh.point_data["density"] = density

    plotter = pv.Plotter()
    plotter.add_mesh(
        mesh,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": density_key,
        },
    )

    plotter.add_text(
        f"Surface density: {density_key}",
        font_size=12,
    )

    plotter.show_axes()
    plotter.show()

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv


def show_fem_distributions(
    verification,
    *,
    case_name,
    stress_limit=None,
    bins=30,
    show_edges=False,
    output_dir="FEM_Maps",
    clim_stress=None,
    clim_displacement=None,
):
    fe = verification.embedded_fem.fe
    mesh = fe.mesh

    # Stress: maximum of 8 integration points per active element
    stress = (
        fe.stress_vm_ip_active
        .detach()
        .cpu()
        .numpy()
        .max(axis=1)
    )
    element_ids = np.asarray(mesh.active_element_ids, dtype=np.int64)

    # Displacement magnitude: one value per mesh node
    u = (
        fe.u.detach()
        .cpu()
        .numpy()
        .reshape(mesh.numNodes, 3)
    )
    displacement = np.linalg.norm(u, axis=1)
    active_node_ids = np.asarray(mesh.active_node_ids, dtype=np.int64)

    if len(stress) != len(element_ids):
        raise ValueError("Stress values do not match the active elements.")

    # Build a grid containing active C3D8 elements only
    connectivity = mesh.elemNodes[element_ids].astype(np.int64)
    cells = np.column_stack([
        np.full(len(element_ids), 8, dtype=np.int64),
        connectivity,
    ]).ravel()

    grid = pv.UnstructuredGrid(
        cells,
        np.full(len(element_ids), pv.CellType.HEXAHEDRON, dtype=np.uint8),
        np.asarray(mesh.nodeXYZ),
    )

    stress_name = "Max IP von Mises (MPa)"
    displacement_name = "Displacement magnitude (mm)"

    grid.cell_data[stress_name] = stress
    grid.point_data[displacement_name] = displacement
    surface = grid.extract_surface()

    # Two maps, with the same camera position
    plotter = pv.Plotter(shape=(1, 2), window_size=(1600, 700))

    Clim_stress_Def = None

    stress_range = (
        (0.0, float(stress.max()))
        if clim_stress is None
        else tuple(clim_stress)
    )

    displacement_range = (
        (0.0, float(displacement[active_node_ids].max()))
        if clim_displacement is None
        else tuple(clim_displacement)
    )

    plotter.subplot(0, 0)
    plotter.add_mesh(
        surface,
        scalars=stress_name,
        preference="cell",
        cmap="turbo",
        clim=stress_range,
        show_edges=show_edges,
        scalar_bar_args={"title": stress_name},
    )
    plotter.add_text(f"{case_name}: stress", font_size=11)
    plotter.add_axes()

    plotter.subplot(0, 1)
    plotter.add_mesh(
        surface,
        scalars=displacement_name,
        preference="point",
        cmap="viridis",
        clim=displacement_range,
        show_edges=show_edges,
        scalar_bar_args={"title": displacement_name},
    )
    plotter.add_text(f"{case_name}: displacement", font_size=11)
    plotter.add_axes()

    plotter.link_views()
    plotter.reset_camera()

    # Histograms: one count per active element and per active node
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].hist(stress, bins=bins)
    if stress_limit is not None:
        axes[0].axvline(
            stress_limit,
            color="red",
            linestyle="--",
            label=f"Limit: {stress_limit:g} MPa",
        )
        axes[0].legend()
    axes[0].set_xlabel(stress_name)
    axes[0].set_ylabel("Number of active elements")
    axes[0].set_title("Element stress distribution")

    axes[1].hist(displacement[active_node_ids], bins=bins)
    axes[1].set_xlabel(displacement_name)
    axes[1].set_ylabel("Number of active nodes")
    axes[1].set_title("Nodal displacement distribution")

    fig.suptitle(case_name)
    fig.tight_layout()

    # Save each model under its own name
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    grid.save(output_dir / f"{case_name}_fields.vtu")
    fig.savefig(output_dir / f"{case_name}_histograms.png", dpi=200)

    plotter.show(screenshot=str(output_dir / f"{case_name}_maps.png"))
    plt.show()

    worst_element_rank = int(np.argmax(stress))
    worst_node = int(active_node_ids[np.argmax(displacement[active_node_ids])])

    print(f"Maximum stress: {stress.max():.4f} MPa")
    print(f"  Element centre: {mesh.elemCenters[element_ids[worst_element_rank]]}")
    print(f"Maximum displacement: {displacement[worst_node]:.6f} mm")
    print(f"  Node position: {mesh.nodeXYZ[worst_node]}")

    if stress_limit is not None:
        above = int(np.count_nonzero(stress > stress_limit))
        print(
            f"Elements above {stress_limit:g} MPa: "
            f"{above}/{len(stress)} ({100 * above / len(stress):.2f}%)"
        )

    return grid

In [3]:
def show_two_surface_densities(
    face_tensor,
    density_phase1,
    density_phase2,
    *,
    title_phase1="Phase 1 - Continuous",
    title_phase2="Phase 2 - Graph",
    binary=False,
    threshold=0.5,
):
    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary
    if triangles.min() == 1:
        triangles = triangles - 1

    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    # --------------------------------------------------------
    # Densities
    # --------------------------------------------------------

    rho1 = to_numpy(density_phase1).reshape(-1)
    rho2 = to_numpy(density_phase2).reshape(-1)

    if len(rho1) != len(points):
        raise ValueError(
            f"Phase-1 density has {len(rho1)} values "
            f"but surface has {len(points)} vertices."
        )

    if len(rho2) != len(points):
        raise ValueError(
            f"Phase-2 density has {len(rho2)} values "
            f"but surface has {len(points)} vertices."
        )

    if binary:
        rho1 = (rho1 > threshold).astype(float)
        rho2 = (rho2 > threshold).astype(float)

    # --------------------------------------------------------
    # Separate meshes so the scalar fields do not overwrite
    # each other
    # --------------------------------------------------------

    mesh1 = pv.PolyData(points, pv_faces)
    mesh2 = pv.PolyData(points, pv_faces)

    mesh1.point_data["density"] = rho1
    mesh2.point_data["density"] = rho2

    # --------------------------------------------------------
    # Side-by-side visualization
    # --------------------------------------------------------

    plotter = pv.Plotter(
        shape=(1, 2),
        window_size=(1700, 750),
    )

    # Phase 1
    plotter.subplot(0, 0)

    plotter.add_mesh(
        mesh1,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": "Density",
        },
    )

    plotter.add_text(
        title_phase1,
        font_size=12,
    )

    plotter.add_axes()

    # Phase 2
    plotter.subplot(0, 1)

    plotter.add_mesh(
        mesh2,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": "Density",
        },
    )

    plotter.add_text(
        title_phase2,
        font_size=12,
    )

    plotter.add_axes()

    # Same camera for both
    plotter.link_views()
    plotter.reset_camera()

    plotter.show()

    return mesh1, mesh2

In [4]:
viz = CADVisualizer()
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Planar = TesPartsDir / "Planar.stp"
FreeFormSurf3 = TesPartsDir / "FreeForm3.stp"
FreeSharp2 = TesPartsDir / "FreeSharp2.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Helmet = TesPartsDir / "Helmet_New.stp"
FreeForm_Bend = TesPartsDir / "FreeForm_bend.stp"
CircBracket_Half = TesPartsDir / "CircBracket_Half.stp"
HelmetNew_Half = TesPartsDir / "HelmetNew_Half.stp"
Wing_NACA2412 = TesPartsDir / "Wing_NACA2412.stp"

aircraft_radome = TesPartsDir / "aircraft_radome.stp"

shape_path = Planar
Face_Cad = CADTensorGenerator(
    device=device,
    seed_domain_mask_res=512,
    mesh_size_scale = 0.5,
)

cad_domain, face_mesh = Face_Cad.generate_from_file(shape_path)
dx,dy,dz = Face_Cad.print_face_info();

points = face_mesh["points_xyz"].detach().cpu().numpy()
faces = face_mesh["pv_faces"].detach().cpu().numpy()

mesh = pv.PolyData(points, faces)

plotter = pv.Plotter()
plotter.add_mesh(
    mesh,
    show_edges=True,
)
plotter.add_axes()
plotter.show()

points_uv  = face_mesh["uv"]
points_xyz = face_mesh["points_xyz"]
Xu         = face_mesh["Xu"]
Xv         = face_mesh["Xv"]
faces_ijk = face_mesh["faces_ijk"].long()

p0 = points_xyz[faces_ijk[:, 0]]
p1 = points_xyz[faces_ijk[:, 1]]
p2 = points_xyz[faces_ijk[:, 2]]

face_areas = 0.5 * torch.linalg.vector_norm(
    torch.cross(
        p1 - p0,
        p2 - p0,
        dim=1,
    ),
    dim=1,
)

surface_area_weights = torch.zeros(
    points_xyz.shape[0],
    device=points_xyz.device,
    dtype=points_xyz.dtype,
)

third_area = face_areas / 3.0

for lv in range(3):
    surface_area_weights.scatter_add_(
        0,
        faces_ijk[:, lv],
        third_area,
    )

print(
    "Surface area:",
    float(surface_area_weights.sum())
)

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    :  - Label 'Shapes/Document' (2D)
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (BSpline)
Info    : [ 30%] Meshing curve 2 (BSpline)
Info    : [ 60%] Meshing curve 3 (BSpline)
Info    : [ 80%] Meshing curve 4 (BSpline)
Info    : Done meshing 1D (Wall 0.000430721s, CPU 0.000464s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.196502s, CPU 0.192436s)
Info    : 12920 nodes 25842 elements
Info    : Clearing all models and views...
Info    : Done clearing all models and views

=== Active Face Info ===
XYZ bounds:
  X: [-0.000000, 10.000000]  span=10.000000
  Y: [-0.000000, 10.000000]  span=10.000000
  Z: [-0.000000, 0.000000]  span=0.000000

UV bounds:
  U: [-10.000000, 0.000000]
  V: [0.000000, 10.000000]

Periodic:
  U periodic: False
  V periodic: False



Widget(value='<iframe src="http://localhost:39743/index.html?ui=P_0x7573c029b370_0&reconnect=auto" class="pyvi…

Surface area: 100.00000762939453


In [5]:
import torch
import numpy as np
import pyvista as pv

from Utils.DifferentiableFilters import smooth_heaviside_projection
def compare_phase1_phase2(
    case_name,
    seeds_uv,
    *,
    tau,

    # ------------------------------------------------------------
    # Geometry
    # ------------------------------------------------------------
    strut_thickness=0.20,
    tube_beta=0.02,

    # ------------------------------------------------------------
    # Phase-1 seed/activity controls
    # ------------------------------------------------------------
    duplicate_merge_sigma=0.02,

    # ------------------------------------------------------------
    # Density comparison
    # ------------------------------------------------------------
    rho_threshold=0.50,

    # ------------------------------------------------------------
    # Common FEM projection
    # ------------------------------------------------------------
    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,

    # ------------------------------------------------------------
    # Phase-1 boundary attachment
    # ------------------------------------------------------------
    boundary_attach_width=2e-5,
    boundary_attach_beta=1e-5,
    boundary_attach_alpha=1.0,

    # ------------------------------------------------------------
    # Fibre visualization
    # ------------------------------------------------------------
    fiber_density_threshold=0.50,
    max_fiber_arrows=700,
    fiber_arrow_scale=0.025,

    visualize=True,
):

    # ============================================================
    # Common geometry
    # ============================================================

    seeds_uv = seeds_uv.to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    S = int(seeds_uv.shape[0])

    strut_thickness = float(strut_thickness)

    # Phase 1 takes radius / half-width.
    strut_radius = 0.5 * strut_thickness

    print("\n" + "=" * 78)
    print(f"CASE: {case_name}")
    print(f"Seeds                 : {S}")
    print(f"Full strut thickness  : {strut_thickness:.6f}")
    print(f"Strut radius          : {strut_radius:.6f}")
    print("=" * 78)

    # ============================================================
    # PHASE 1
    # Implicit continuous Voronoi decoder
    # ============================================================

    phase1_decoder = Phase1VoronoiDecoder(

        n_seeds=S,

        use_Metric_anisotropy=False,

        # P1 expects radius, not full width.
        fixed_strut_radius=strut_radius,

        # Legacy output requirement only.
        # Not used to define in-plane strut width.
        fixed_height=1.0,

        physical_tube_beta=float(
            tube_beta
        ),

        duplicate_merge_sigma=float(
            duplicate_merge_sigma
        ),

        territory_min_ratio=0.05,

        continuous_length_calibration=1.0,

        use_boundary_attachment=True,

        point_chunk_size=1024,

        boundary_attach_width=float(
            boundary_attach_width
        ),

        boundary_attach_beta=float(
            boundary_attach_beta
        ),

        boundary_attach_alpha=float(
            boundary_attach_alpha
        ),

        # No internal projection.
        density_projection_strength=0.0,

    ).to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # Required by P1 API.
    # Does not control radius because fixed_strut_radius is set.
    w_raw = torch.zeros(
        S,
        S,
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # ============================================================
    # PHASE 2
    # Explicit Voronoi graph decoder
    # ============================================================

    phase2_decoder = ContinuousVoronoiDecoder(

        Cad_domain=Face_Cad,

        face_mesh=face_mesh,

        n_seeds=S,

        # P2 expects FULL width.
        strut_thickness=strut_thickness,

        beta=float(
            tube_beta
        ),

        tube_curve_samples=64,

        tube_distance_tau=float(
            tube_beta
        ),

        tube_density_tau=float(
            tube_beta
        ),

        tube_fiber_tau=float(
            tube_beta
        ),

        rho_min=0.0,

    ).to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # ============================================================
    # Evaluate
    # ============================================================

    with torch.no_grad():

        out_p1 = phase1_decoder(

            points_uv=points_uv,

            Xu=Xu,

            Xv=Xv,

            tau=float(
                tau
            ),

            seeds_raw=seeds_uv,

            w_raw=w_raw,

            h_raw=None,

            points_face_id=points_face_id,

            boundary_uv=boundary_uv,

            boundary_face_id=boundary_face_id,

            boundary_curve_offsets=boundary_curve_offsets,

            surface_area_weights=surface_area_weights,
        )

        out_p2 = phase2_decoder(

            seeds_uv=seeds_uv,

            generate_density_fiber=True,
        )

    # ============================================================
    # Raw density + fibre
    # ============================================================

    rho1_raw = out_p1["rho"]

    rho2_raw = out_p2["rho"]

    fiber1 = out_p1["fiber3d"]

    fiber2 = out_p2["fiber3d"]

    # Normalize fibre directions.
    fiber1 = torch.nn.functional.normalize(
        fiber1,
        dim=-1,
        eps=1e-8,
    )

    fiber2 = torch.nn.functional.normalize(
        fiber2,
        dim=-1,
        eps=1e-8,
    )

    # ============================================================
    # Common FEM-style density projection
    # ============================================================

    rho1_proj = smooth_heaviside_projection(

        rho1_raw,

        beta=float(
            projection_beta
        ),

        eta=float(
            projection_eta
        ),

        strength=float(
            projection_strength
        ),

        eps=1e-8,

        debug=False,
    )

    rho2_proj = smooth_heaviside_projection(

        rho2_raw,

        beta=float(
            projection_beta
        ),

        eta=float(
            projection_eta
        ),

        strength=float(
            projection_strength
        ),

        eps=1e-8,

        debug=False,
    )

    # ============================================================
    # Binary fields
    # Only for geometry comparison / visualization
    # ============================================================

    rho1_bin = (
        rho1_proj >= rho_threshold
    ).to(
        rho1_proj.dtype
    )

    rho2_bin = (
        rho2_proj >= rho_threshold
    ).to(
        rho2_proj.dtype
    )

    # ============================================================
    # Nominal thickness diagnostics
    # ============================================================

    print("\n--- Nominal thickness ---")

    print(
        "Requested full thickness :",
        strut_thickness,
    )

    print(
        "P1 imposed radius        :",
        strut_radius,
    )

    print(
        "P1 implied full thickness:",
        2.0 * strut_radius,
    )

    print(
        "P2 reported radius       :",
        float(
            out_p2[
                "centerline_radius"
            ]
        ),
    )

    print(
        "P2 full thickness        :",
        float(
            out_p2[
                "strut_thickness"
            ]
        ),
    )

    # ============================================================
    # Density ranges
    # ============================================================

    def print_density_range(
        name,
        rho,
    ):

        print(
            f"{name:<14}: "
            f"min={float(rho.min()):.6f}  "
            f"mean={float(rho.mean()):.6f}  "
            f"max={float(rho.max()):.6f}"
        )

    print(
        "\n--- Density ranges ---"
    )

    print_density_range(
        "P1 raw",
        rho1_raw,
    )

    print_density_range(
        "P2 raw",
        rho2_raw,
    )

    print_density_range(
        "P1 projected",
        rho1_proj,
    )

    print_density_range(
        "P2 projected",
        rho2_proj,
    )

    # ============================================================
    # LENGTH
    # ============================================================

    L1 = out_p1[
        "continuous_voronoi_length"
    ]

    curves_xyz = out_p2[
        "edge_curves_xyz"
    ]

    segment_vectors = (
        curves_xyz[:, 1:, :]
        -
        curves_xyz[:, :-1, :]
    )

    segment_lengths = (
        torch.linalg.vector_norm(
            segment_vectors,
            dim=-1,
        )
    )

    edge_lengths = (
        segment_lengths.sum(
            dim=1
        )
    )

    graph = out_p2[
        "graph"
    ]

    edge_type = graph[
        "edge_type"
    ]

    # Type 4 = domain shell boundary.
    internal_mask = (
        edge_type != 4
    )

    boundary_mask = (
        edge_type == 4
    )

    L2_internal = (
        edge_lengths[
            internal_mask
        ].sum()
    )

    L2_boundary = (
        edge_lengths[
            boundary_mask
        ].sum()
    )

    L2_total = (
        edge_lengths.sum()
    )

    length_error_pct = (

        100.0

        * torch.abs(
            L1
            -
            L2_internal
        )

        / L2_internal
        .abs()
        .clamp_min(
            1e-8
        )
    )

    print(
        "\n--- Length ---"
    )

    print(
        f"P1 continuous Voronoi : "
        f"{float(L1):.6f}"
    )

    print(
        f"P2 internal Voronoi   : "
        f"{float(L2_internal):.6f}"
    )

    print(
        f"P2 boundary shell     : "
        f"{float(L2_boundary):.6f}"
    )

    print(
        f"P2 total              : "
        f"{float(L2_total):.6f}"
    )

    print(
        f"P1/P2 internal error  : "
        f"{float(length_error_pct):.3f} %"
    )

    # ============================================================
    # Material area helper
    # ============================================================

    thresholds = [
        0.01,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
    ]

    def material_area(
        rho,
        threshold,
    ):

        return (

            (
                rho >= threshold
            )
            .to(
                surface_area_weights.dtype
            )

            * surface_area_weights

        ).sum()

    # ============================================================
    # RAW AREA SWEEP
    # ============================================================

    print(
        "\n--- Raw density area sweep ---"
    )

    for thr in thresholds:

        A1 = material_area(
            rho1_raw,
            thr,
        )

        A2 = material_area(
            rho2_raw,
            thr,
        )

        if float(A2) > 1e-8:

            ratio_text = (
                f"{float(A1 / A2):.4f}"
            )

        else:

            ratio_text = "N/A"

        print(
            f"rho >= {thr:4.2f}: "
            f"P1={float(A1):9.4f}   "
            f"P2={float(A2):9.4f}   "
            f"ratio={ratio_text}"
        )

    # ============================================================
    # PROJECTED AREA SWEEP
    # ============================================================

    print(
        "\n--- FEM-projected area sweep ---"
    )

    for thr in thresholds:

        A1 = material_area(
            rho1_proj,
            thr,
        )

        A2 = material_area(
            rho2_proj,
            thr,
        )

        if float(A2) > 1e-8:

            ratio_text = (
                f"{float(A1 / A2):.4f}"
            )

        else:

            ratio_text = "N/A"

        print(
            f"rho >= {thr:4.2f}: "
            f"P1={float(A1):9.4f}   "
            f"P2={float(A2):9.4f}   "
            f"ratio={ratio_text}"
        )

    # ============================================================
    # Projected density errors
    # ============================================================

    rho_diff = (
        rho1_proj
        -
        rho2_proj
    )

    rho_mae = (
        rho_diff
        .abs()
        .mean()
    )

    rho_rmse = torch.sqrt(
        rho_diff
        .pow(2)
        .mean()
    )

    weighted_rho_mae = (

        (
            rho_diff.abs()
            *
            surface_area_weights
        ).sum()

        /

        surface_area_weights
        .sum()
        .clamp_min(
            1e-8
        )
    )

    print(
        "\n--- FEM-projected density error ---"
    )

    print(
        "MAE          :",
        float(rho_mae),
    )

    print(
        "RMSE         :",
        float(rho_rmse),
    )

    print(
        "Weighted MAE :",
        float(weighted_rho_mae),
    )

    # ============================================================
    # Binary geometry comparison
    # ============================================================

    A1_bin = (
        rho1_bin
        *
        surface_area_weights
    ).sum()

    A2_bin = (
        rho2_bin
        *
        surface_area_weights
    ).sum()

    binary_area_ratio = (
        A1_bin
        /
        A2_bin.clamp_min(
            1e-8
        )
    )

    print(
        "\n--- Final binary geometry @ rho=0.5 ---"
    )

    print(
        "P1 binary area:",
        float(A1_bin),
    )

    print(
        "P2 binary area:",
        float(A2_bin),
    )

    print(
        "P1/P2 ratio :",
        float(binary_area_ratio),
    )

    # ============================================================
    # Fibre comparison
    #
    # Fibre is axial:
    #
    #       t == -t
    #
    # therefore use |dot|.
    # ============================================================

    axial_dot = torch.abs(

        (
            fiber1
            *
            fiber2
        ).sum(
            dim=-1
        )

    ).clamp(
        0.0,
        1.0,
    )

    angle_deg = (

        torch.acos(
            axial_dot
        )

        * 180.0

        / torch.pi
    )

    # Compare only where BOTH projected
    # representations contain material.
    common_material = (

        (
            rho1_proj
            >= rho_threshold
        )

        &

        (
            rho2_proj
            >= rho_threshold
        )
    )

    if bool(
        common_material.any()
    ):

        common_angles = (
            angle_deg[
                common_material
            ]
        )

        fiber_angle_mean = (
            common_angles.mean()
        )

        fiber_angle_max = (
            common_angles.max()
        )

        fiber_angle_p95 = torch.quantile(
            common_angles,
            0.95,
        )

    else:

        fiber_angle_mean = (
            angle_deg.new_tensor(
                float("nan")
            )
        )

        fiber_angle_max = (
            angle_deg.new_tensor(
                float("nan")
            )
        )

        fiber_angle_p95 = (
            angle_deg.new_tensor(
                float("nan")
            )
        )

    print(
        "\n--- Fibre field ---"
    )

    print(
        "Mean axial angle error:",
        float(
            fiber_angle_mean
        ),
        "deg",
    )

    print(
        "95th percentile error :",
        float(
            fiber_angle_p95
        ),
        "deg",
    )

    print(
        "Max axial angle error :",
        float(
            fiber_angle_max
        ),
        "deg",
    )

    # ============================================================
    # Topology
    # ============================================================

    phase2_seed_mask = (
        out_p1.get(
            "phase2_seed_mask",
            None,
        )
    )

    if (
        phase2_seed_mask
        is not None
    ):

        handoff_count = int(

            phase2_seed_mask
            .sum()
            .detach()
            .cpu()
        )

    else:

        handoff_count = S

    print(
        "\n--- Topology ---"
    )

    print(
        "Original seeds      :",
        S,
    )

    print(
        "P1 handoff seeds    :",
        handoff_count,
    )

    print(
        "P2 graph edge count :",
        int(
            curves_xyz.shape[0]
        ),
    )

    # ============================================================
    # Density parameters
    # ============================================================

    local_scale = (

        phase2_decoder
        ._local_uv_to_xyz_scale(

            Xu,
            Xv,
            points_xyz,
        )
    )

    p2_tau_density = (

        phase2_decoder
        .tube_density_tau

        *
        local_scale
    )

    print(
        "\n--- Density parameters ---"
    )

    print(
        "P1 physical radius:",
        strut_radius,
    )

    print(
        "P1 physical beta  :",
        float(
            phase1_decoder
            .physical_tube_beta
        ),
    )

    print(
        "P2 radius         :",
        float(
            out_p2[
                "centerline_radius"
            ]
        ),
    )

    print(
        "P2 local scale    :",
        float(
            local_scale
        ),
    )

    print(
        "P2 effective beta :",
        float(
            p2_tau_density
        ),
    )

    # ============================================================
    # VISUALIZATION
    # ============================================================

    if visualize:

        xyz_np = (
            points_xyz
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        tri_np = (
            faces_ijk
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        n_faces = (
            tri_np.shape[0]
        )

        pv_faces = np.hstack(
            [
                np.full(
                    (
                        n_faces,
                        1,
                    ),
                    3,
                    dtype=np.int64,
                ),

                tri_np,
            ]
        ).reshape(-1)

        # --------------------------------------------------------
        # Convert fields
        # --------------------------------------------------------

        rho1_raw_np = (
            rho1_raw
            .detach()
            .cpu()
            .numpy()
        )

        rho2_raw_np = (
            rho2_raw
            .detach()
            .cpu()
            .numpy()
        )

        rho1_proj_np = (
            rho1_proj
            .detach()
            .cpu()
            .numpy()
        )

        rho2_proj_np = (
            rho2_proj
            .detach()
            .cpu()
            .numpy()
        )

        rho1_bin_np = (
            rho1_bin
            .detach()
            .cpu()
            .numpy()
        )

        rho2_bin_np = (
            rho2_bin
            .detach()
            .cpu()
            .numpy()
        )

        fiber1_np = (
            fiber1
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        fiber2_np = (
            fiber2
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        # ========================================================
        # Helper: build mesh
        # ========================================================

        def make_mesh(
            scalar_field,
        ):

            mesh = pv.PolyData(
                xyz_np,
                pv_faces,
            )

            mesh["rho"] = (
                scalar_field.astype(
                    np.float32
                )
            )

            return mesh

        # ========================================================
        # Helper: add fibre arrows
        # ========================================================

        def add_fiber_vectors(
            plotter,
            density,
            fiber_vectors,
            *,
            threshold,
            max_arrows,
            scale_factor,
        ):

            density = np.asarray(
                density
            )

            fiber_vectors = np.asarray(
                fiber_vectors
            )

            fiber_norm = np.linalg.norm(
                fiber_vectors,
                axis=1,
            )

            valid = (

                np.isfinite(
                    density
                )

                &

                np.isfinite(
                    fiber_vectors
                ).all(
                    axis=1
                )

                &

                (
                    density
                    >= threshold
                )

                &

                (
                    fiber_norm
                    > 1e-10
                )
            )

            idx = np.flatnonzero(
                valid
            )

            if idx.size == 0:

                return 0

            # --------------------------------------------
            # Subsample arrows for readability
            # --------------------------------------------

            if idx.size > max_arrows:

                stride = int(
                    np.ceil(
                        idx.size
                        /
                        max_arrows
                    )
                )

                idx = idx[
                    ::stride
                ]

            arrow_points = (
                xyz_np[
                    idx
                ]
            )

            arrow_vectors = (
                fiber_vectors[
                    idx
                ]
            )

            # Normalize once more before glyphing.
            arrow_vectors = (

                arrow_vectors

                /

                np.linalg.norm(
                    arrow_vectors,
                    axis=1,
                    keepdims=True,
                )
            )

            # --------------------------------------------
            # Physical arrow size relative to model
            # --------------------------------------------

            bbox = (
                xyz_np.max(
                    axis=0
                )
                -
                xyz_np.min(
                    axis=0
                )
            )

            diagonal = np.linalg.norm(
                bbox
            )

            arrow_length = (
                scale_factor
                *
                diagonal
            )

            cloud = pv.PolyData(
                arrow_points
            )

            cloud[
                "fiber"
            ] = arrow_vectors

            glyphs = cloud.glyph(

                orient="fiber",

                scale=False,

                factor=float(
                    arrow_length
                ),

                geom=pv.Arrow(),
            )

            plotter.add_mesh(
                glyphs,
            )

            return int(
                idx.size
            )

        # ========================================================
        # 3D comparison
        #
        # Row 1: raw densities
        # Row 2: projected densities + fibre vectors
        # Row 3: binary geometries
        # ========================================================

        plotter = pv.Plotter(

            shape=(3, 2),

            window_size=(
                1500,
                1500,
            ),
        )

        # --------------------------------------------------------
        # P1 RAW
        # --------------------------------------------------------

        plotter.subplot(
            0,
            0,
        )

        mesh = make_mesh(
            rho1_raw_np
        )

        plotter.add_text(
            "Phase 1 — raw density",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P2 RAW
        # --------------------------------------------------------

        plotter.subplot(
            0,
            1,
        )

        mesh = make_mesh(
            rho2_raw_np
        )

        plotter.add_text(
            "Phase 2 — raw density",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P1 PROJECTED + FIBRES
        # --------------------------------------------------------

        plotter.subplot(
            1,
            0,
        )

        mesh = make_mesh(
            rho1_proj_np
        )

        plotter.add_text(
            "Phase 1 — FEM projected + fibres",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            opacity=0.75,

            show_edges=False,
        )

        p1_arrow_count = add_fiber_vectors(

            plotter,

            rho1_proj_np,

            fiber1_np,

            threshold=float(
                fiber_density_threshold
            ),

            max_arrows=int(
                max_fiber_arrows
            ),

            scale_factor=float(
                fiber_arrow_scale
            ),
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P2 PROJECTED + FIBRES
        # --------------------------------------------------------

        plotter.subplot(
            1,
            1,
        )

        mesh = make_mesh(
            rho2_proj_np
        )

        plotter.add_text(
            "Phase 2 — FEM projected + fibres",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            opacity=0.75,

            show_edges=False,
        )

        p2_arrow_count = add_fiber_vectors(

            plotter,

            rho2_proj_np,

            fiber2_np,

            threshold=float(
                fiber_density_threshold
            ),

            max_arrows=int(
                max_fiber_arrows
            ),

            scale_factor=float(
                fiber_arrow_scale
            ),
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P1 BINARY
        # --------------------------------------------------------

        plotter.subplot(
            2,
            0,
        )

        mesh = make_mesh(
            rho1_bin_np
        )

        plotter.add_text(
            "Phase 1 — binary geometry",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P2 BINARY
        # --------------------------------------------------------

        plotter.subplot(
            2,
            1,
        )

        mesh = make_mesh(
            rho2_bin_np
        )

        plotter.add_text(
            "Phase 2 — binary geometry",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # Same view everywhere
        # --------------------------------------------------------

        plotter.link_views()

        plotter.show()

        print(
            "\n--- Fibre visualization ---"
        )

        print(
            "P1 arrows shown:",
            p1_arrow_count,
        )

        print(
            "P2 arrows shown:",
            p2_arrow_count,
        )

        print(
            "Fibre density threshold:",
            fiber_density_threshold,
        )

    # ============================================================
    # Return metrics
    # ============================================================

    metrics = {

        "case":
            case_name,

        "seed_count":
            S,

        "strut_thickness":
            strut_thickness,

        "strut_radius":
            strut_radius,

        "P1_length":
            float(
                L1
            ),

        "P2_internal_length":
            float(
                L2_internal
            ),

        "P2_boundary_length":
            float(
                L2_boundary
            ),

        "P2_total_length":
            float(
                L2_total
            ),

        "length_error_pct":
            float(
                length_error_pct
            ),

        "projected_rho_mae":
            float(
                rho_mae
            ),

        "projected_rho_rmse":
            float(
                rho_rmse
            ),

        "projected_weighted_rho_mae":
            float(
                weighted_rho_mae
            ),

        "P1_binary_area":
            float(
                A1_bin
            ),

        "P2_binary_area":
            float(
                A2_bin
            ),

        "binary_area_ratio":
            float(
                binary_area_ratio
            ),

        "fiber_angle_mean_deg":
            float(
                fiber_angle_mean
            ),

        "fiber_angle_p95_deg":
            float(
                fiber_angle_p95
            ),

        "fiber_angle_max_deg":
            float(
                fiber_angle_max
            ),

        "P1_handoff_seeds":
            handoff_count,

        "P2_edge_count":
            int(
                curves_xyz.shape[0]
            ),
    }

    return (
        metrics,
        out_p1,
        out_p2,
    )



In [6]:
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

In [7]:
# ============================================================
# Common CAD data
# ============================================================

points_uv  = face_mesh["uv"]
points_xyz = face_mesh["points_xyz"]
Xu         = face_mesh["Xu"]
Xv         = face_mesh["Xv"]
faces_ijk  = face_mesh["faces_ijk"].long()

# Robust face-ID lookup
points_face_id = face_mesh.get(
    "points_face_id",
    face_mesh.get("face_id", None),
)

boundary_uv = face_mesh.get(
    "boundary_curve_uv",
    face_mesh.get("boundary_uv", None),
)

boundary_face_id = face_mesh.get(
    "boundary_face_id",
    None,
)

boundary_curve_offsets = face_mesh.get(
    "boundary_curve_offsets",
    None,
)

In [8]:
test_seed_cases15 = {
    "handoff_overcomplete": random_seeds_min_dist(
        N=80,
        min_dist=0.035,
        seed=727,
        device=device,
    ),
}

In [11]:
test_seed_cases14 = {
    "nonuniform_cell_sizes": torch.tensor(
[[0.9219, 0.4948],
        [0.0787, 0.0762],
        
        [0.2149, 0.9289],
        [0.9131, 0.0475],
        [0.4079, 0.5705],
        [0.5955, 0.0792],
        [0.6416, 0.9304],
        [0.9272, 0.7717],
        [0.9148, 0.2424],
        [0.5655, 0.4302],
        [0.2521, 0.7608],
        [0.2499, 0.2553],
        [0.7426, 0.3806],
        [0.9243, 0.6294],
        [0.7867, 0.9317],
        [0.5864, 0.7610],
        [0.0765, 0.7552],
        [0.0691, 0.4188],
        [0.7431, 0.5134],
        [0.4299, 0.0707],
        [0.7515, 0.2317],
        [0.3606, 0.9216],
        [0.9250, 0.3640],
        [0.7546, 0.7802],
        [0.9409, 0.1149],
        [0.3972, 0.4012],
        [0.5861, 0.2594],
        [0.0640, 0.9254],
        [0.0725, 0.5880],
        [0.4206, 0.7424],
        [0.0759, 0.2499],
        [0.9359, 0.9300],
        [0.4990, 0.9233],
        [0.5766, 0.5973],
        [0.7494, 0.6457],
        [0.4229, 0.2370],
        [0.2334, 0.4311],
        [0.7547, 0.0711],
        [0.2573, 0.0774],
        [0.2432, 0.5992]], 
        dtype=torch.float32,
        device=device,
    ),
}

In [15]:
test_seed_cases_close = {
    "with_physical_duplicates": torch.tensor(
        [
            [0.10, 0.10],

            # close pair: 0.40 physical
            [0.30, 0.30],
            [0.34, 0.30],

            # close diagonal pair: ~0.424 physical
            [0.55, 0.20],
            [0.58, 0.23],

            # close 3-point cluster
            [0.70, 0.70],
            [0.72, 0.70],   # 0.20 physical
            [0.71, 0.73],   # close to both

            # well-separated points
            [0.15, 0.75],
            [0.25, 0.55],
            [0.42, 0.82],
            [0.50, 0.50],
            [0.62, 0.42],
            [0.82, 0.18],
            [0.88, 0.48],
            [0.85, 0.82],

            # another close pair: 0.30 physical
            [0.18, 0.40],
            [0.21, 0.40],

            # another close diagonal pair: ~0.283 physical
            [0.42, 0.18],
            [0.44, 0.20],
        ],
        dtype=torch.float32,
        device=device,
    ),
}

In [19]:
results = []

test_seed_cases11 = {

    "normal_random": random_seeds_min_dist(
        N=50,
        min_dist=0.1,
        seed=344,
        device=device,
    ),
}


for case_name, seeds_uv in test_seed_cases_close.items():

    try:

        metrics, out_p1, out_p2 = compare_phase1_phase2(

            case_name,
            seeds_uv,

    tau=0.01,

    strut_thickness=0.4,
    tube_beta=0.01,

    duplicate_merge_sigma=0.3,

   


    projection_beta=8.0,
    projection_eta=0.5,
    projection_strength=1.0,

    visualize=True,
        )

        results.append(
            metrics
        )

    except Exception as exc:

        print(
            f"\nFAILED: {case_name}"
        )

        print(
            type(exc).__name__,
            str(exc),
        )


CASE: with_physical_duplicates
Seeds                 : 20
Full strut thickness  : 0.400000
Strut radius          : 0.200000

--- Nominal thickness ---
Requested full thickness : 0.4
P1 imposed radius        : 0.2
P1 implied full thickness: 0.4
P2 reported radius       : 0.20000000298023224
P2 full thickness        : 0.4000000059604645

--- Density ranges ---
P1 raw        : min=0.000000  mean=0.356934  max=1.000000
P2 raw        : min=0.000000  mean=0.327441  max=0.880692
P1 projected  : min=0.000000  mean=0.340265  max=1.000000
P2 projected  : min=0.000000  mean=0.332744  max=0.998076

--- Length ---
P1 continuous Voronoi : 61.291046
P2 internal Voronoi   : 68.674454
P2 boundary shell     : 40.000000
P2 total              : 108.674454
P1/P2 internal error  : 10.751 %

--- Raw density area sweep ---
rho >= 0.01: P1=  90.6499   P2=  75.2330   ratio=1.2049
rho >= 0.10: P1=  37.9999   P2=  59.8649   ratio=0.6348
rho >= 0.25: P1=  34.6976   P2=  46.6115   ratio=0.7444
rho >= 0.50: P1=  32

Widget(value='<iframe src="http://localhost:39743/index.html?ui=P_0x756db8473d60_6&reconnect=auto" class="pyvi…


--- Fibre visualization ---
P1 arrows shown: 628
P2 arrows shown: 618
Fibre density threshold: 0.5


In [ ]:
results = []

test_seed_cases11 = {

    "normal_random": random_seeds_min_dist(
        N=50,
        min_dist=0.1,
        seed=344,
        device=device,
    ),
}


for case_name, seeds_uv in test_seed_cases14.items():

    try:

        metrics, out_p1, out_p2 = compare_phase1_phase2(

            case_name,
            seeds_uv,

            tau=0.01,

            strut_thickness=0.2,

            tube_beta=0.01,

            duplicate_merge_sigma=0.1,

            rho_threshold=0.50,

            projection_beta=8.0,
            projection_eta=0.5,
            projection_strength=1.0,

            boundary_attach_width=2e-5,
            boundary_attach_beta=1e-5,
            boundary_attach_alpha=1.0,

            # Fibre arrows
            fiber_density_threshold=0.50,
            max_fiber_arrows=5000,
            fiber_arrow_scale=0.005,

            visualize=True,
        )

        results.append(
            metrics
        )

    except Exception as exc:

        print(
            f"\nFAILED: {case_name}"
        )

        print(
            type(exc).__name__,
            str(exc),
        )

In [ ]:
voxel_size = float((dx + dy + dz) / 200.0)
thickness = 3.0 * voxel_size
print(f"dx:{dx} ,dy:{dy} ,dz:{dz}")
print(f"voxel size : {voxel_size}")
print(f"thickness : {thickness}")
LoadingCase ="ReaLCase"
common_settings = dict(
    thickness=thickness,
    voxel_size=voxel_size,
    extra_layers=1,
    tensors=face_mesh,

    voxelization_mode="triangle_distance",
    subvoxel_samples=1,
    min_geom_fraction=0.25,

    faces_index_base=0,
)

shell_problem = ThickenShell(
    **common_settings
)
Boundary_loads = [

    {
        "type": "force",
        "boundaries": [2,3],
        "magnitude": 1,
        "direction": "z",
    },
    {
        "type": "distributed_force",
        "boundaries": [0,1],
        "magnitude": 5.0,
        "direction": "y",
        "distribution": "parabolic",
    },
]

shell_problem.show_boundaries(
    segments_per_curve=2
)
BC_Report = shell_problem.boundary_loading(
    fixed=[6,7],
    loads=Boundary_loads,
    band=0.05* voxel_size,
)

In [ ]:
loading_img = shell_problem.show_voxels_surface_and_bc(
    return_img=True,
    off_screen=True,
    window_size=(560, 430),
    show=True,
)

In [ ]:
import torch
import numpy as np
import pyvista as pv

from Utils.DifferentiableFilters import smooth_heaviside_projection


def compare_phase1_phase2_FEM(
    case_name,
    seeds_uv,
    verification,
    *,
    tau,

    # ------------------------------------------------------------
    # Geometry
    # ------------------------------------------------------------
    strut_thickness=0.20,
    tube_beta=0.01,

    # ------------------------------------------------------------
    # Phase-1 controls
    # ------------------------------------------------------------
    duplicate_merge_sigma=0.02,

    boundary_attach_width=2e-5,
    boundary_attach_beta=1e-5,
    boundary_attach_alpha=1.0,

    # ------------------------------------------------------------
    # Density
    # ------------------------------------------------------------
    rho_threshold=0.50,

    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,

    # ------------------------------------------------------------
    # FEA
    # ------------------------------------------------------------
    run_fea=True,
    fea_material_bins=2,
    material_mass_density_kg_m3=1550.0,
    stress_limit=300.0,

    # ------------------------------------------------------------
    # Fibre visualization
    # ------------------------------------------------------------
    fiber_density_threshold=0.50,
    max_fiber_arrows=1000,
    fiber_arrow_scale=0.015,

    visualize=True,
):

    # ============================================================
    # COMMON SETUP
    # ============================================================

    seeds_uv = seeds_uv.to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    S = int(seeds_uv.shape[0])

    strut_thickness = float(strut_thickness)
    strut_radius = 0.5 * strut_thickness

    print("\n" + "=" * 80)
    print(f"CASE: {case_name}")
    print(f"Seeds                 : {S}")
    print(f"Full strut thickness  : {strut_thickness:.6f}")
    print(f"Strut radius          : {strut_radius:.6f}")
    print("=" * 80)

    # ============================================================
    # PHASE 1
    # ============================================================

    phase1_decoder = Phase1VoronoiDecoder(

        n_seeds=S,

        use_Metric_anisotropy=False,

        # P1 uses radius / half-width
        fixed_strut_radius=strut_radius,

        # Legacy value required by current P1 API.
        # Does NOT control the in-plane strut width.
        fixed_height=1.0,

        physical_tube_beta=float(tube_beta),

        duplicate_merge_sigma=float(
            duplicate_merge_sigma
        ),

        territory_min_ratio=0.05,

        continuous_length_calibration=1.0,

        use_boundary_attachment=True,

        point_chunk_size=1024,

        boundary_attach_width=float(
            boundary_attach_width
        ),

        boundary_attach_beta=float(
            boundary_attach_beta
        ),

        boundary_attach_alpha=float(
            boundary_attach_alpha
        ),

        # Common projection is performed outside the decoder
        density_projection_strength=0.0,

    ).to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    w_raw = torch.zeros(
        S,
        S,
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # ============================================================
    # PHASE 2
    # ============================================================

    phase2_decoder = ContinuousVoronoiDecoder(

        Cad_domain=Face_Cad,

        face_mesh=face_mesh,

        n_seeds=S,

        # P2 uses complete strut thickness
        strut_thickness=strut_thickness,

        beta=float(tube_beta),

        tube_curve_samples=64,

        tube_distance_tau=float(tube_beta),
        tube_density_tau=float(tube_beta),
        tube_fiber_tau=float(tube_beta),

        rho_min=0.0,

    ).to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # ============================================================
    # DECODE
    # ============================================================

    with torch.no_grad():

        out_p1 = phase1_decoder(

            points_uv=points_uv,
            Xu=Xu,
            Xv=Xv,

            tau=float(tau),

            seeds_raw=seeds_uv,

            w_raw=w_raw,
            h_raw=None,

            points_face_id=points_face_id,

            boundary_uv=boundary_uv,
            boundary_face_id=boundary_face_id,
            boundary_curve_offsets=boundary_curve_offsets,

            surface_area_weights=surface_area_weights,
        )

        out_p2 = phase2_decoder(

            seeds_uv=seeds_uv,

            generate_density_fiber=True,
        )

    # ============================================================
    # RAW FIELDS
    # ============================================================

    rho1_raw = out_p1["rho"]

    rho2_raw = out_p2["rho"]

    fiber1 = torch.nn.functional.normalize(
        out_p1["fiber3d"],
        dim=-1,
        eps=1e-8,
    )

    fiber2 = torch.nn.functional.normalize(
        out_p2["fiber3d"],
        dim=-1,
        eps=1e-8,
    )

    # ============================================================
    # COMMON FEM PROJECTION
    # ============================================================

    rho1_projected = smooth_heaviside_projection(

        rho1_raw,

        beta=float(projection_beta),

        eta=float(projection_eta),

        strength=float(projection_strength),

        eps=1e-8,

        debug=False,
    )

    rho2_projected = smooth_heaviside_projection(

        rho2_raw,

        beta=float(projection_beta),

        eta=float(projection_eta),

        strength=float(projection_strength),

        eps=1e-8,

        debug=False,
    )

    # ============================================================
    # BINARY REPRESENTATIONS
    # ============================================================

    rho1_raw_binary = (
        rho1_raw >= rho_threshold
    ).to(rho1_raw.dtype)

    rho2_raw_binary = (
        rho2_raw >= rho_threshold
    ).to(rho2_raw.dtype)

    rho1_projected_binary = (
        rho1_projected >= rho_threshold
    ).to(rho1_projected.dtype)

    rho2_projected_binary = (
        rho2_projected >= rho_threshold
    ).to(rho2_projected.dtype)

    # ============================================================
    # THICKNESS CHECK
    # ============================================================

    print("\n--- Nominal thickness ---")

    print(
        "Requested full thickness :",
        strut_thickness,
    )

    print(
        "P1 imposed radius        :",
        strut_radius,
    )

    print(
        "P1 implied full thickness:",
        2.0 * strut_radius,
    )

    print(
        "P2 reported radius       :",
        float(
            out_p2["centerline_radius"]
        ),
    )

    print(
        "P2 full thickness        :",
        float(
            out_p2["strut_thickness"]
        ),
    )

    # ============================================================
    # LENGTH COMPARISON
    # ============================================================

    L1 = out_p1[
        "continuous_voronoi_length"
    ]

    curves_xyz = out_p2[
        "edge_curves_xyz"
    ]

    segment_vectors = (
        curves_xyz[:, 1:, :]
        -
        curves_xyz[:, :-1, :]
    )

    edge_lengths = (
        torch.linalg.vector_norm(
            segment_vectors,
            dim=-1,
        ).sum(dim=1)
    )

    edge_type = out_p2[
        "graph"
    ]["edge_type"]

    internal_mask = (
        edge_type != 4
    )

    boundary_mask = (
        edge_type == 4
    )

    L2_internal = (
        edge_lengths[
            internal_mask
        ].sum()
    )

    L2_boundary = (
        edge_lengths[
            boundary_mask
        ].sum()
    )

    L2_total = (
        edge_lengths.sum()
    )

    length_error_pct = (

        100.0

        * torch.abs(
            L1 - L2_internal
        )

        / L2_internal
        .abs()
        .clamp_min(1e-8)
    )

    print("\n--- Length ---")

    print(
        f"P1 continuous Voronoi : "
        f"{float(L1):.6f}"
    )

    print(
        f"P2 internal Voronoi   : "
        f"{float(L2_internal):.6f}"
    )

    print(
        f"P2 boundary shell     : "
        f"{float(L2_boundary):.6f}"
    )

    print(
        f"P2 total              : "
        f"{float(L2_total):.6f}"
    )

    print(
        f"P1/P2 internal error  : "
        f"{float(length_error_pct):.3f} %"
    )

    # ============================================================
    # DENSITY ERROR
    # ============================================================

    projected_diff = (
        rho1_projected
        -
        rho2_projected
    )

    projected_mae = (
        projected_diff
        .abs()
        .mean()
    )

    projected_rmse = torch.sqrt(
        projected_diff
        .pow(2)
        .mean()
    )

    weighted_projected_mae = (

        (
            projected_diff.abs()
            *
            surface_area_weights
        ).sum()

        /

        surface_area_weights
        .sum()
        .clamp_min(1e-8)
    )

    print(
        "\n--- Projected density difference ---"
    )

    print(
        "MAE          :",
        float(projected_mae),
    )

    print(
        "RMSE         :",
        float(projected_rmse),
    )

    print(
        "Weighted MAE :",
        float(weighted_projected_mae),
    )

    # ============================================================
    # MATERIAL AREAS
    # ============================================================

    def area_of_density(
        rho,
        threshold,
    ):

        return (

            (
                rho >= threshold
            )
            .to(
                surface_area_weights.dtype
            )

            *
            surface_area_weights

        ).sum()

    thresholds = [
        0.01,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
    ]

    print(
        "\n--- Projected area sweep ---"
    )

    for threshold in thresholds:

        A1 = area_of_density(
            rho1_projected,
            threshold,
        )

        A2 = area_of_density(
            rho2_projected,
            threshold,
        )

        if float(A2) > 1e-12:

            ratio = (
                float(A1 / A2)
            )

            ratio_text = (
                f"{ratio:.4f}"
            )

        else:

            ratio_text = "N/A"

        print(
            f"rho >= {threshold:4.2f}: "
            f"P1={float(A1):9.4f}   "
            f"P2={float(A2):9.4f}   "
            f"ratio={ratio_text}"
        )

    # ============================================================
    # FIBRE COMPARISON
    # ============================================================

    axial_dot = torch.abs(

        (
            fiber1
            *
            fiber2
        ).sum(
            dim=-1
        )

    ).clamp(
        0.0,
        1.0,
    )

    angle_deg = (

        torch.acos(
            axial_dot
        )

        *
        180.0

        /
        torch.pi
    )

    common_material = (

        (
            rho1_projected
            >= rho_threshold
        )

        &

        (
            rho2_projected
            >= rho_threshold
        )
    )

    if bool(
        common_material.any()
    ):

        angles_common = (
            angle_deg[
                common_material
            ]
        )

        fiber_angle_mean = (
            angles_common.mean()
        )

        fiber_angle_p95 = (
            torch.quantile(
                angles_common,
                0.95,
            )
        )

        fiber_angle_max = (
            angles_common.max()
        )

    else:

        nan = angle_deg.new_tensor(
            float("nan")
        )

        fiber_angle_mean = nan
        fiber_angle_p95 = nan
        fiber_angle_max = nan

    print(
        "\n--- Fibre field ---"
    )

    print(
        "Mean axial angle error:",
        float(
            fiber_angle_mean
        ),
        "deg",
    )

    print(
        "95th percentile error :",
        float(
            fiber_angle_p95
        ),
        "deg",
    )

    print(
        "Max axial angle error :",
        float(
            fiber_angle_max
        ),
        "deg",
    )

    # ============================================================
    # FEA CASE DEFINITIONS
    # ============================================================

    fea_results = {}

    if run_fea:

        density_cases = {

            # --------------------------------------------
            # Phase 1
            # --------------------------------------------

            "P1_raw":
                (
                    rho1_raw,
                    fiber1,
                ),

            "P1_projected":
                (
                    rho1_projected,
                    fiber1,
                ),

            "P1_raw_binary":
                (
                    rho1_raw_binary,
                    fiber1,
                ),

            "P1_projected_binary":
                (
                    rho1_projected_binary,
                    fiber1,
                ),

            # --------------------------------------------
            # Phase 2
            # --------------------------------------------

            "P2_raw":
                (
                    rho2_raw,
                    fiber2,
                ),

            "P2_projected":
                (
                    rho2_projected,
                    fiber2,
                ),

            "P2_raw_binary":
                (
                    rho2_raw_binary,
                    fiber2,
                ),

            "P2_projected_binary":
                (
                    rho2_projected_binary,
                    fiber2,
                ),
        }

        # ========================================================
        # Helper to run one FEA case
        # ========================================================

        def run_fea_case(
            label,
            rho,
            fiber,
        ):

            fields = {

                "rho":
                    rho,

                "density":
                    rho,

                "fiber3d":
                    fiber,

                "fiber":
                    fiber,
            }

            fem_case_name = (
                f"{case_name}_{label}"
            )

            metrics = verification.run_fields(

                fields=fields,

                case_name=fem_case_name,

                material_bins=int(
                    fea_material_bins
                ),

                material_mass_density_kg_m3=float(
                    material_mass_density_kg_m3
                ),
            )

            # --------------------------------------------
            # Additional stress-limit information
            # --------------------------------------------

            stress_ip = (

                verification
                .embedded_fem
                .fe
                .stress_vm_ip_active
                .detach()
                .cpu()
                .numpy()
            )

            # Maximum integration-point stress per element
            stress_element_max = (
                stress_ip.max(
                    axis=1
                )
            )

            above_limit_elements = int(
                np.count_nonzero(
                    stress_element_max
                    >
                    float(stress_limit)
                )
            )

            metrics = dict(
                metrics
            )

            metrics[
                "elements_above_stress_limit"
            ] = above_limit_elements

            metrics[
                "stress_limit"
            ] = float(
                stress_limit
            )

            return metrics

        # ========================================================
        # Run all representations
        # ========================================================

        print(
            "\n"
            + "=" * 80
        )

        print(
            "FEA REPRESENTATION COMPARISON"
        )

        print(
            "=" * 80
        )

        for (
            label,
            (
                rho_case,
                fiber_case,
            ),
        ) in density_cases.items():

            print(
                f"\n>>> Running {label}"
            )

            fea_results[
                label
            ] = run_fea_case(

                label,

                rho_case,

                fiber_case,
            )

        # ========================================================
        # Compact FEA table
        # ========================================================

        print(
            "\n"
            + "=" * 125
        )

        print(
            "FEA SUMMARY"
        )

        print(
            "=" * 125
        )

        header = (
            f"{'Case':<24}"
            f"{'Max U':>14}"
            f"{'Compliance':>16}"
            f"{'Max VM':>16}"
            f"{'> limit':>12}"
            f"{'Mass [g]':>16}"
        )

        print(
            header
        )

        print(
            "-" * len(header)
        )

        for (
            label,
            fem_metrics,
        ) in fea_results.items():

            max_u = fem_metrics[
                "global_max_displacement_magnitude"
            ]

            compliance = fem_metrics[
                "compliance"
            ]

            max_stress = fem_metrics[
                "maximum_integration_point_von_mises"
            ]

            above_limit = fem_metrics[
                "elements_above_stress_limit"
            ]

            mass = fem_metrics.get(
                "embedded_total_mass_grams",
                float("nan"),
            )

            print(

                f"{label:<24}"

                f"{max_u:>14.6g}"

                f"{compliance:>16.6g}"

                f"{max_stress:>16.6g}"

                f"{above_limit:>12d}"

                f"{mass:>16.6g}"
            )

        # ========================================================
        # Direct P1/P2 ratios for each representation
        # ========================================================

        print(
            "\n--- P1 / P2 structural ratios ---"
        )

        pair_names = [

            (
                "raw",
                "P1_raw",
                "P2_raw",
            ),

            (
                "projected",
                "P1_projected",
                "P2_projected",
            ),

            (
                "raw_binary",
                "P1_raw_binary",
                "P2_raw_binary",
            ),

            (
                "projected_binary",
                "P1_projected_binary",
                "P2_projected_binary",
            ),
        ]

        for (
            representation,
            p1_name,
            p2_name,
        ) in pair_names:

            m1 = fea_results[
                p1_name
            ]

            m2 = fea_results[
                p2_name
            ]

            displacement_ratio = (

                m1[
                    "global_max_displacement_magnitude"
                ]

                /

                max(
                    m2[
                        "global_max_displacement_magnitude"
                    ],
                    1e-12,
                )
            )

            compliance_ratio = (

                m1[
                    "compliance"
                ]

                /

                max(
                    m2[
                        "compliance"
                    ],
                    1e-12,
                )
            )

            stress_ratio = (

                m1[
                    "maximum_integration_point_von_mises"
                ]

                /

                max(
                    m2[
                        "maximum_integration_point_von_mises"
                    ],
                    1e-12,
                )
            )

            print(
                f"\n{representation}:"
            )

            print(
                f"  displacement P1/P2 : "
                f"{displacement_ratio:.4f}"
            )

            print(
                f"  compliance P1/P2   : "
                f"{compliance_ratio:.4f}"
            )

            print(
                f"  max stress P1/P2   : "
                f"{stress_ratio:.4f}"
            )

    # ============================================================
    # TOPOLOGY
    # ============================================================

    phase2_seed_mask = out_p1.get(
        "phase2_seed_mask",
        None,
    )

    if phase2_seed_mask is not None:

        handoff_count = int(

            phase2_seed_mask
            .sum()
            .detach()
            .cpu()
        )

    else:

        handoff_count = S

    print(
        "\n--- Topology ---"
    )

    print(
        "Original seeds      :",
        S,
    )

    print(
        "P1 handoff seeds    :",
        handoff_count,
    )

    print(
        "P2 graph edge count :",
        int(
            curves_xyz.shape[0]
        ),
    )

    # ============================================================
    # DENSITY PARAMETERS
    # ============================================================

    local_scale = (

        phase2_decoder
        ._local_uv_to_xyz_scale(

            Xu,
            Xv,
            points_xyz,
        )
    )

    p2_effective_beta = (

        phase2_decoder
        .tube_density_tau

        *
        local_scale
    )

    print(
        "\n--- Density parameters ---"
    )

    print(
        "P1 physical radius:",
        strut_radius,
    )

    print(
        "P1 physical beta  :",
        float(
            phase1_decoder
            .physical_tube_beta
        ),
    )

    print(
        "P2 radius         :",
        float(
            out_p2[
                "centerline_radius"
            ]
        ),
    )

    print(
        "P2 local scale    :",
        float(
            local_scale
        ),
    )

    print(
        "P2 effective beta :",
        float(
            p2_effective_beta
        ),
    )

    # ============================================================
    # 3D VISUALIZATION
    # ============================================================

    if visualize:

        xyz_np = (
            points_xyz
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        tri_np = (
            faces_ijk
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        number_of_faces = (
            tri_np.shape[0]
        )

        pv_faces = np.hstack(
            [
                np.full(
                    (
                        number_of_faces,
                        1,
                    ),
                    3,
                    dtype=np.int64,
                ),

                tri_np,
            ]
        ).reshape(-1)

        def make_mesh(
            rho,
        ):

            mesh = pv.PolyData(
                xyz_np,
                pv_faces,
            )

            mesh[
                "rho"
            ] = (

                rho
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            return mesh

        def add_fibre_arrows(
            plotter,
            rho,
            fibre,
        ):

            rho_np = (
                rho
                .detach()
                .cpu()
                .numpy()
            )

            fibre_np = (
                fibre
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            norms = np.linalg.norm(
                fibre_np,
                axis=1,
            )

            mask = (

                np.isfinite(
                    rho_np
                )

                &

                np.isfinite(
                    fibre_np
                ).all(
                    axis=1
                )

                &

                (
                    rho_np
                    >=
                    float(
                        fiber_density_threshold
                    )
                )

                &

                (
                    norms
                    >
                    1e-10
                )
            )

            ids = np.flatnonzero(
                mask
            )

            if ids.size == 0:

                return 0

            if (
                ids.size
                >
                int(
                    max_fiber_arrows
                )
            ):

                stride = int(
                    np.ceil(

                        ids.size

                        /

                        int(
                            max_fiber_arrows
                        )
                    )
                )

                ids = ids[
                    ::stride
                ]

            vectors = (
                fibre_np[
                    ids
                ]
            )

            vectors /= np.linalg.norm(
                vectors,
                axis=1,
                keepdims=True,
            )

            cloud = pv.PolyData(
                xyz_np[
                    ids
                ]
            )

            cloud[
                "fiber"
            ] = vectors

            bbox = (
                xyz_np.max(
                    axis=0
                )
                -
                xyz_np.min(
                    axis=0
                )
            )

            diagonal = (
                np.linalg.norm(
                    bbox
                )
            )

            glyphs = cloud.glyph(

                orient="fiber",

                scale=False,

                factor=float(
                    fiber_arrow_scale
                    *
                    diagonal
                ),

                geom=pv.Arrow(),
            )

            plotter.add_mesh(
                glyphs
            )

            return int(
                ids.size
            )

        # ========================================================
        # 3 x 2 figure
        # ========================================================

        plotter = pv.Plotter(

            shape=(3, 2),

            window_size=(
                1600,
                1400,
            ),
        )

        visual_fields = [

            (
                0,
                0,
                "P1 raw density",
                rho1_raw,
                None,
            ),

            (
                0,
                1,
                "P2 raw density",
                rho2_raw,
                None,
            ),

            (
                1,
                0,
                "P1 projected + fibre",
                rho1_projected,
                fiber1,
            ),

            (
                1,
                1,
                "P2 projected + fibre",
                rho2_projected,
                fiber2,
            ),

            (
                2,
                0,
                "P1 binary",
                rho1_projected_binary,
                None,
            ),

            (
                2,
                1,
                "P2 binary",
                rho2_projected_binary,
                None,
            ),
        ]

        for (
            row,
            col,
            title,
            rho,
            fibre,
        ) in visual_fields:

            plotter.subplot(
                row,
                col,
            )

            mesh = make_mesh(
                rho
            )

            plotter.add_text(
                title,
                font_size=11,
            )

            plotter.add_mesh(

                mesh,

                scalars="rho",

                clim=[
                    0.0,
                    1.0,
                ],

                opacity=(
                    0.70
                    if fibre is not None
                    else 1.0
                ),

                show_edges=False,
            )

            if fibre is not None:

                add_fibre_arrows(
                    plotter,
                    rho,
                    fibre,
                )

            plotter.add_axes()

            plotter.camera_position = (
                "iso"
            )

        plotter.link_views()

        plotter.show()

    # ============================================================
    # FINAL RETURN
    # ============================================================

    metrics = {

        "case":
            case_name,

        "seed_count":
            S,

        "strut_thickness":
            strut_thickness,

        "strut_radius":
            strut_radius,

        "P1_length":
            float(
                L1
            ),

        "P2_internal_length":
            float(
                L2_internal
            ),

        "length_error_pct":
            float(
                length_error_pct
            ),

        "projected_rho_mae":
            float(
                projected_mae
            ),

        "projected_rho_rmse":
            float(
                projected_rmse
            ),

        "projected_weighted_rho_mae":
            float(
                weighted_projected_mae
            ),

        "fiber_angle_mean_deg":
            float(
                fiber_angle_mean
            ),

        "fiber_angle_p95_deg":
            float(
                fiber_angle_p95
            ),

        "fiber_angle_max_deg":
            float(
                fiber_angle_max
            ),

        "P1_handoff_seeds":
            handoff_count,

        "P2_edge_count":
            int(
                curves_xyz.shape[0]
            ),

        "FEA":
            fea_results,
    }

    return (
        metrics,
        out_p1,
        out_p2,
        fea_results,
    )

In [ ]:
verification = VoxelFEMAbaqusRunner(
    face_mesh=face_mesh,
    shell_problem=shell_problem,
    device=device,
    output_dir="FEM_Verification",
)
LoadingCase= "bend"

In [ ]:
test_seed_cases12 = {
    "difficult_irregular": torch.tensor(
        [
            # --------------------------------------------------
            # Sparse upper-left / background distribution
            # --------------------------------------------------
            [0.06, 0.08],
            [0.18, 0.10],
            [0.34, 0.07],
            [0.55, 0.10],
            [0.78, 0.08],
            [0.94, 0.12],

            [0.08, 0.25],
            [0.24, 0.22],
            [0.43, 0.26],
            [0.67, 0.23],
            [0.90, 0.27],

            # --------------------------------------------------
            # Dense but NON-duplicated central region
            # --------------------------------------------------
            [0.38, 0.38],
            [0.45, 0.36],
            [0.53, 0.39],
            [0.61, 0.36],

            [0.40, 0.45],
            [0.47, 0.43],
            [0.55, 0.46],
            [0.63, 0.44],

            [0.37, 0.53],
            [0.45, 0.51],
            [0.53, 0.54],
            [0.61, 0.52],

            [0.42, 0.60],
            [0.50, 0.59],
            [0.58, 0.61],

            # --------------------------------------------------
            # Irregular right-side distribution
            # --------------------------------------------------
            [0.72, 0.38],
            [0.84, 0.42],
            [0.93, 0.36],

            [0.70, 0.56],
            [0.82, 0.60],
            [0.95, 0.54],

            # --------------------------------------------------
            # Lower region with uneven spacing
            # --------------------------------------------------
            [0.07, 0.68],
            [0.21, 0.64],
            [0.34, 0.72],
            [0.48, 0.69],
            [0.66, 0.73],
            [0.86, 0.70],

            [0.10, 0.86],
            [0.27, 0.91],
            [0.46, 0.84],
            [0.62, 0.90],
            [0.79, 0.85],
            [0.93, 0.92],

            # --------------------------------------------------
            # Boundary-near, but still valid seeds
            # --------------------------------------------------
            [0.02, 0.40],
            [0.025, 0.58],
            [0.975, 0.18],
            [0.98, 0.76],

            [0.18, 0.025],
            [0.73, 0.025],
        ],
        dtype=torch.float32,
        device=device,
    ),
}

In [ ]:
test_seed_cases13 = {
    "boundary_heavy": torch.tensor(
        [
            # --------------------------------------------------
            # Near lower boundary
            # --------------------------------------------------
            [0.05, 0.03],
            [0.16, 0.025],
            [0.29, 0.035],
            [0.43, 0.020],
            [0.57, 0.030],
            [0.71, 0.025],
            [0.84, 0.035],
            [0.95, 0.030],

            # --------------------------------------------------
            # Near upper boundary
            # --------------------------------------------------
            [0.06, 0.965],
            [0.18, 0.975],
            [0.31, 0.960],
            [0.45, 0.980],
            [0.59, 0.970],
            [0.73, 0.975],
            [0.86, 0.960],
            [0.96, 0.970],

            # --------------------------------------------------
            # Near left boundary
            # --------------------------------------------------
            [0.025, 0.16],
            [0.030, 0.31],
            [0.020, 0.47],
            [0.035, 0.64],
            [0.025, 0.82],

            # --------------------------------------------------
            # Near right boundary
            # --------------------------------------------------
            [0.975, 0.14],
            [0.970, 0.30],
            [0.980, 0.49],
            [0.965, 0.66],
            [0.975, 0.84],

            # --------------------------------------------------
            # Interior support seeds
            # --------------------------------------------------
            [0.18, 0.20],
            [0.36, 0.18],
            [0.55, 0.22],
            [0.76, 0.20],

            [0.20, 0.40],
            [0.39, 0.38],
            [0.58, 0.42],
            [0.79, 0.39],

            [0.17, 0.60],
            [0.35, 0.63],
            [0.55, 0.59],
            [0.77, 0.62],

            [0.22, 0.80],
            [0.41, 0.82],
            [0.61, 0.79],
            [0.80, 0.81],

            # --------------------------------------------------
            # Extra irregular interior seeds
            # --------------------------------------------------
            [0.28, 0.51],
            [0.47, 0.50],
            [0.68, 0.50],
            [0.50, 0.31],
            [0.50, 0.69],
            [0.33, 0.72],
            [0.67, 0.28],
            [0.84, 0.53],
        ],
        dtype=torch.float32,
        device=device,
    ),
}

In [ ]:
test_seed_cases14 = {
    "nonuniform_cell_sizes": torch.tensor(
        [
            # --------------------------------------------------
            # Sparse left region
            # --------------------------------------------------
            [0.08, 0.10],
            [0.12, 0.38],
            [0.09, 0.70],
            [0.16, 0.91],

            [0.28, 0.16],
            [0.25, 0.52],
            [0.31, 0.84],

            # --------------------------------------------------
            # Moderately populated centre-left
            # --------------------------------------------------
            [0.40, 0.10],
            [0.42, 0.28],
            [0.39, 0.47],
            [0.43, 0.68],
            [0.40, 0.90],

            # --------------------------------------------------
            # Dense centre-right cluster
            # Distinct seeds, no near-duplicates
            # --------------------------------------------------
            [0.55, 0.22],
            [0.61, 0.20],
            [0.67, 0.23],
            [0.73, 0.21],

            [0.53, 0.31],
            [0.59, 0.30],
            [0.65, 0.33],
            [0.71, 0.30],
            [0.77, 0.32],

            [0.55, 0.41],
            [0.61, 0.40],
            [0.67, 0.43],
            [0.73, 0.40],

            [0.53, 0.51],
            [0.59, 0.50],
            [0.65, 0.53],
            [0.71, 0.50],
            [0.77, 0.52],

            [0.55, 0.61],
            [0.61, 0.60],
            [0.67, 0.63],
            [0.73, 0.60],

            # --------------------------------------------------
            # Sparse upper-right region
            # --------------------------------------------------
            [0.86, 0.12],
            [0.91, 0.30],
            [0.88, 0.54],
            [0.93, 0.78],
            [0.85, 0.92],

            # --------------------------------------------------
            # Additional irregular transition seeds
            # --------------------------------------------------
            [0.48, 0.18],
            [0.47, 0.38],
            [0.49, 0.58],
            [0.46, 0.79],

            [0.79, 0.18],
            [0.82, 0.40],
            [0.80, 0.64],
            [0.77, 0.84],

            [0.22, 0.30],
            [0.20, 0.72],
        ],
        dtype=torch.float32,
        device=device,
    ),
}

In [ ]:
test_seed_cases15 = {
    "handoff_overcomplete": random_seeds_min_dist(
        N=30,
        min_dist=0.035,
        seed=727,
        device=device,
    ),
}

In [ ]:
results = []

for case_name, seeds_uv in test_seed_cases12.items():

    try:

        (
            metrics,
            out_p1,
            out_p2,
            fea_results,
        ) = compare_phase1_phase2_FEM(

            case_name,
            seeds_uv,

            verification=verification,

            tau=0.01,

            strut_thickness=0.30,

            tube_beta=0.01,

            duplicate_merge_sigma=0.1,

            rho_threshold=0.50,

            projection_beta=8.0,
            projection_eta=0.50,
            projection_strength=1.0,

            boundary_attach_width=2e-5,
            boundary_attach_beta=1e-5,
            boundary_attach_alpha=1.0,

            run_fea=True,

            fea_material_bins=2,

            material_mass_density_kg_m3=1550.0,

            stress_limit=300.0,

            fiber_density_threshold=0.50,
            max_fiber_arrows=1000,
            fiber_arrow_scale=0.015,

            visualize=True,
        )

        results.append(
            metrics
        )

    except Exception as exc:

        print(
            f"\nFAILED: {case_name}"
        )

        print(
            type(exc).__name__,
            str(exc),
        )

In [ ]:
def test_handoff_threshold_sweep(
    out_p1,
    *,
    strut_thickness=0.30,
    tube_beta=0.01,
    thresholds=(0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50),
    rho_threshold=0.50,
    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,
):

    activity = out_p1["seed_active_weights"]
    territory = out_p1["seed_territory_fraction"]
    domain_mask = out_p1["seed_domain_hard_mask"]

    # This is the same territory criterion currently used by the decoder.
    territory_threshold = out_p1["phase2_territory_threshold"]

    all_seeds = out_p1["seeds_uv"]

    # ----------------------------------------------------------
    # Phase-1 reference
    # ----------------------------------------------------------

    L1 = out_p1["continuous_voronoi_length"]

    rho1_raw = out_p1["rho"]

    rho1_proj = smooth_heaviside_projection(
        rho1_raw,
        beta=float(projection_beta),
        eta=float(projection_eta),
        strength=float(projection_strength),
        eps=1e-8,
        debug=False,
    )

    A1 = (
        (rho1_proj >= rho_threshold)
        .to(surface_area_weights.dtype)
        * surface_area_weights
    ).sum()

    print("\n" + "=" * 100)
    print("HANDOFF ACTIVITY-THRESHOLD SWEEP")
    print("=" * 100)

    print(f"P1 reference length     : {float(L1):.6f}")
    print(f"P1 projected area @0.5 : {float(A1):.6f}")

    print(
        "\n"
        f"{'thr':>6}"
        f"{'seeds':>8}"
        f"{'L2':>14}"
        f"{'L err %':>12}"
        f"{'A2@0.5':>14}"
        f"{'A1/A2':>12}"
        f"{'rho MAE':>12}"
    )

    print("-" * 78)

    results = []

    for threshold in thresholds:

        # ------------------------------------------------------
        # Same handoff logic, only varying activity threshold
        # ------------------------------------------------------

        activity_ok = (
            activity >= float(threshold)
        )

        territory_ok = (
            territory >= territory_threshold
        )

        mask = (
            domain_mask
            & activity_ok
            & territory_ok
        )

        handoff_seeds = all_seeds[mask]

        n = int(
            handoff_seeds.shape[0]
        )

        if n < 3:

            print(
                f"{threshold:>6.3f}"
                f"{n:>8d}"
                "   insufficient seeds"
            )

            continue

        # ------------------------------------------------------
        # Phase 2
        # ------------------------------------------------------

        decoder_p2 = ContinuousVoronoiDecoder(

            Cad_domain=Face_Cad,

            face_mesh=face_mesh,

            n_seeds=n,

            strut_thickness=float(
                strut_thickness
            ),

            beta=float(
                tube_beta
            ),

            tube_curve_samples=64,

            tube_distance_tau=float(
                tube_beta
            ),

            tube_density_tau=float(
                tube_beta
            ),

            tube_fiber_tau=float(
                tube_beta
            ),

            rho_min=0.0,

        ).to(
            device=points_uv.device,
            dtype=points_uv.dtype,
        )

        with torch.no_grad():

            out_p2_test = decoder_p2(

                seeds_uv=handoff_seeds,

                generate_density_fiber=True,
            )

        # ------------------------------------------------------
        # P2 internal length
        # ------------------------------------------------------

        curves_xyz = out_p2_test[
            "edge_curves_xyz"
        ]

        edge_lengths = (
            torch.linalg.vector_norm(
                curves_xyz[:, 1:, :]
                -
                curves_xyz[:, :-1, :],
                dim=-1,
            )
            .sum(dim=1)
        )

        edge_type = (
            out_p2_test[
                "graph"
            ][
                "edge_type"
            ]
        )

        internal_mask = (
            edge_type != 4
        )

        L2 = (
            edge_lengths[
                internal_mask
            ].sum()
        )

        length_error = (
            100.0
            *
            torch.abs(
                L1 - L2
            )
            /
            L1.abs().clamp_min(
                1e-8
            )
        )

        # ------------------------------------------------------
        # P2 projected density
        # ------------------------------------------------------

        rho2_proj = smooth_heaviside_projection(

            out_p2_test["rho"],

            beta=float(
                projection_beta
            ),

            eta=float(
                projection_eta
            ),

            strength=float(
                projection_strength
            ),

            eps=1e-8,

            debug=False,
        )

        A2 = (
            (rho2_proj >= rho_threshold)
            .to(surface_area_weights.dtype)
            *
            surface_area_weights
        ).sum()

        area_ratio = (
            A1
            /
            A2.clamp_min(
                1e-8
            )
        )

        rho_mae = (
            rho1_proj
            -
            rho2_proj
        ).abs().mean()

        # ------------------------------------------------------
        # Print
        # ------------------------------------------------------

        print(
            f"{threshold:>6.3f}"
            f"{n:>8d}"
            f"{float(L2):>14.3f}"
            f"{float(length_error):>12.2f}"
            f"{float(A2):>14.3f}"
            f"{float(area_ratio):>12.3f}"
            f"{float(rho_mae):>12.3f}"
        )

        results.append(
            {
                "activity_threshold":
                    float(threshold),

                "handoff_seed_count":
                    n,

                "P1_length":
                    float(L1),

                "P2_internal_length":
                    float(L2),

                "length_error_pct":
                    float(length_error),

                "P1_area":
                    float(A1),

                "P2_area":
                    float(A2),

                "area_ratio":
                    float(area_ratio),

                "rho_mae":
                    float(rho_mae),
            }
        )

    return results

In [ ]:
handoff_sweep = test_handoff_threshold_sweep(
    out_p1,

    strut_thickness=0.30,

    tube_beta=0.01,

    thresholds=[
        0.05,
        0.075,
        0.10,
        0.15,
        0.20,
        0.25,
        0.30,
        0.40,
        0.50,
    ],

    rho_threshold=0.50,
)

In [ ]:
import torch
import numpy as np

from Utils.DifferentiableFilters import smooth_heaviside_projection


def final_phase1_gradient_test(
    seeds_uv,
    *,
    tau=0.01,
    strut_thickness=0.30,
    tube_beta=0.01,
    duplicate_merge_sigma=0.02,
    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,
):

    print("\n" + "=" * 95)
    print("FINAL PHASE-1 SINGLE-BACKWARD TEST")
    print("=" * 95)

    # ============================================================
    # Trainable seed coordinates
    # ============================================================

    seeds_test = (
        seeds_uv
        .detach()
        .clone()
        .to(
            device=points_uv.device,
            dtype=points_uv.dtype,
        )
    )

    seeds_test.requires_grad_(True)

    N = int(seeds_test.shape[0])

    radius = 0.5 * float(strut_thickness)

    # ============================================================
    # Decoder
    # ============================================================

    decoder = Phase1VoronoiDecoder(
        n_seeds=N,

        use_Metric_anisotropy=False,

        fixed_strut_radius=radius,

        # Legacy dummy only
        fixed_height=1.0,

        physical_tube_beta=float(tube_beta),

        duplicate_merge_sigma=float(
            duplicate_merge_sigma
        ),

        territory_min_ratio=0.05,

        # Current handoff threshold
        phase2_activity_threshold=0.05,

        continuous_length_calibration=1.0,

        use_boundary_attachment=True,

        # Keep small for memory-sensitive training test
        point_chunk_size=32,

        boundary_attach_width=2e-5,
        boundary_attach_beta=1e-5,
        boundary_attach_alpha=1.0,

        # Projection is applied externally/common to P1 and P2
        density_projection_strength=0.0,

    ).to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    w_raw = torch.zeros(
        N,
        N,
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # ============================================================
    # Forward
    # ============================================================

    out = decoder(
        points_uv=points_uv,

        Xu=Xu,
        Xv=Xv,

        tau=float(tau),

        seeds_raw=seeds_test,

        w_raw=w_raw,

        h_raw=None,

        points_face_id=points_face_id,

        boundary_uv=boundary_uv,

        boundary_face_id=boundary_face_id,

        boundary_curve_offsets=boundary_curve_offsets,

        surface_area_weights=surface_area_weights,
    )

    rho_raw = out["rho"]

    L_cont = out[
        "continuous_voronoi_length"
    ]

    # ============================================================
    # Common smooth projection
    #
    # This is the density representation intended for FEM.
    # ============================================================

    rho_projected = smooth_heaviside_projection(
        rho_raw,

        beta=float(
            projection_beta
        ),

        eta=float(
            projection_eta
        ),

        strength=float(
            projection_strength
        ),

        eps=1e-8,

        debug=False,
    )

    # ============================================================
    # Projected material term
    # ============================================================

    area_weights = surface_area_weights.to(
        device=rho_projected.device,
        dtype=rho_projected.dtype,
    )

    L_density = (
        rho_projected
        *
        area_weights
    ).sum() / (
        area_weights.sum()
        .clamp_min(1e-8)
    )

    # ============================================================
    # Diagnostic helper
    # ============================================================

    def analyse_gradient(
        name,
        grad,
    ):

        grad_flat = grad.reshape(-1)

        finite = bool(
            torch.isfinite(
                grad_flat
            ).all()
            .detach()
        )

        abs_grad = (
            grad_flat
            .abs()
        )

        nonzero = int(
            (
                abs_grad > 1e-12
            )
            .sum()
            .detach()
        )

        total = int(
            grad_flat.numel()
        )

        norm = float(
            torch.linalg.vector_norm(
                grad_flat
            )
            .detach()
        )

        max_abs = float(
            abs_grad.max()
            .detach()
        )

        mean_abs = float(
            abs_grad.mean()
            .detach()
        )

        print(
            f"\n{name}"
        )

        print(
            "  finite             :",
            finite,
        )

        print(
            "  gradient norm      :",
            norm,
        )

        print(
            "  mean |grad|        :",
            mean_abs,
        )

        print(
            "  max  |grad|        :",
            max_abs,
        )

        print(
            "  nonzero components :",
            f"{nonzero}/{total}",
        )

        return {
            "finite": finite,
            "norm": norm,
            "mean_abs": mean_abs,
            "max_abs": max_abs,
            "nonzero": nonzero,
            "total": total,
        }

    # ============================================================
    # ONE representative Phase-1 training objective
    #
    # Normalization is only for this diagnostic so that the
    # numerical magnitude of one term does not dominate the other.
    # ============================================================

    L_test = (
        L_cont
        /
        L_cont.detach()
        .abs()
        .clamp_min(1e-8)

        +

        L_density
        /
        L_density.detach()
        .abs()
        .clamp_min(1e-8)
    )

    print(
        "\n--- Objective values ---"
    )

    print(
        "Continuous length :",
        float(
            L_cont.detach()
        ),
    )

    print(
        "Projected density :",
        float(
            L_density.detach()
        ),
    )

    print(
        "Combined test loss:",
        float(
            L_test.detach()
        ),
    )

    # ============================================================
    # SINGLE BACKWARD
    # ============================================================

    grad = torch.autograd.grad(
        L_test,

        seeds_test,

        retain_graph=False,

        create_graph=False,

        allow_unused=False,
    )[0]

    grad_stats = analyse_gradient(
        "Single Phase-1 objective -> seeds",
        grad,
    )

    # ============================================================
    # Per-seed gradient diagnostics
    # ============================================================

    per_seed_grad = torch.linalg.vector_norm(
        grad,
        dim=-1,
    )

    active_gradient_mask = (
        per_seed_grad > 1e-10
    )

    print(
        "\n--- Per-seed gradient ---"
    )

    print(
        "min  :",
        float(
            per_seed_grad.min()
            .detach()
        ),
    )

    print(
        "mean :",
        float(
            per_seed_grad.mean()
            .detach()
        ),
    )

    print(
        "max  :",
        float(
            per_seed_grad.max()
            .detach()
        ),
    )

    print(
        "seeds with |grad| > 1e-10:",
        int(
            active_gradient_mask
            .sum()
            .detach()
        ),
        "/",
        N,
    )

    # ============================================================
    # Activity / handoff diagnostics
    # ============================================================

    activity = (
        out[
            "seed_active_weights"
        ]
        .detach()
    )

    handoff = (
        out[
            "phase2_seed_mask"
        ]
        .detach()
    )

    print(
        "\n--- Activity / handoff ---"
    )

    print(
        "Initial seeds       :",
        N,
    )

    print(
        "Soft active count   :",
        float(
            out[
                "soft_active_seed_count"
            ]
            .detach()
        ),
    )

    print(
        "Handoff count       :",
        int(
            handoff
            .sum()
            .detach()
        ),
    )

    print(
        "Activity min/max    :",
        float(
            activity.min()
        ),
        "/",
        float(
            activity.max()
        ),
    )

    # ============================================================
    # Final automatic check
    # ============================================================

    passed = (
        grad_stats["finite"]
        and
        grad_stats["norm"] > 0.0
        and
        grad_stats["nonzero"] > 0
        and
        bool(
            torch.isfinite(
                rho_projected
            )
            .all()
            .detach()
        )
    )

    print(
        "\n" + "=" * 95
    )

    if passed:

        print(
            "RESULT: PASS — single-backward Phase-1 training path works."
        )

    else:

        print(
            "RESULT: FAIL — inspect gradient diagnostics before trainer integration."
        )

    print(
        "=" * 95
    )

    return {
        "passed":
            passed,

        "gradient":
            grad_stats,

        "per_seed_gradient":
            per_seed_grad.detach(),

        "output":
            out,

        "rho_projected":
            rho_projected.detach(),

        "continuous_length":
            L_cont.detach(),

        "projected_density":
            L_density.detach(),
    }

In [ ]:
test_seed_cases15 = {
    "handoff_overcomplete": random_seeds_min_dist(
        N=80,
        min_dist=0.05,
        seed=727,
        device=device,
    ),
}

print(
    test_seed_cases15[
        "handoff_overcomplete"
    ].shape
)

gradient_test = final_phase1_gradient_test(
    test_seed_cases15[
        "handoff_overcomplete"
    ],

    tau=0.01,

    strut_thickness=0.30,

    tube_beta=0.01,

    duplicate_merge_sigma=0.02,

    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,
)

In [ ]:
test_seed_cases15 = {
    "handoff_overcomplete": random_seeds_min_dist(
        N=50,
        min_dist=0.05,
        seed=727,
        device=device,
    ),
}

print(
    test_seed_cases15[
        "handoff_overcomplete"
    ].shape
)

gradient_test = final_phase1_gradient_test(
    test_seed_cases15[
        "handoff_overcomplete"
    ],

    tau=0.01,

    strut_thickness=0.30,

    tube_beta=0.01,

    duplicate_merge_sigma=0.02,

    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,
)

In [ ]:
Strut_Thickness = 0.4
decoder = ContinuousVoronoiDecoder(
    Cad_domain=Face_Cad,
    face_mesh=face_mesh,
    n_seeds=100,
    strut_thickness=Strut_Thickness,
    seed_domain_margin=0.8,
    # Keep your other settings unchanged.
)
baseline = NearUniformHoneycombBaseline(
    decoder=decoder,
    cad_domain=Face_Cad,
    face_mesh=face_mesh,

    target_seed_count=None,
    spacing_uv=0.170,

    mode="uv_hex",
    relaxation_steps=0,
    relaxation_factor=0.5,

    include_exterior_support_sites=True,
    support_band_cells=2.5,

    relax_support_sites=True,
    support_relaxation_neighbors=6,
    max_relaxation_displacement_cells=0.75,
    preserve_lattice_topology=True,
)

honeycomb_fields = baseline.generate(
    device=device,
    dtype=torch.float64,
    generate_density_fiber=True,
)

curves_uv = honeycomb_fields["edge_curves_uv"]          # [E, K, 2]
curves_xyz = honeycomb_fields["edge_curves_xyz"]        # [E, K, 3]
graph = honeycomb_fields["graph"]
honeycomb_fields["rho"] = ( honeycomb_fields["rho"] > 0.5).to(dtype=honeycomb_fields["rho"].dtype,device=honeycomb_fields["rho"].device,)
total_all = honeycomb_fields["total_curve_length"]                 # scalar tensor
total_vd = honeycomb_fields["total_voronoi_curve_length"] 
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))
show_surface_density(honeycomb_fields, density_key="rho")





In [ ]:
#honeycomb_fields["rho"] = ( honeycomb_fields["rho"] > 0.5).to(dtype=honeycomb_fields["rho"].dtype,device=honeycomb_fields["rho"].device,)
optimized_metrics = verification.run_fields(
    fields=honeycomb_fields,
    case_name="LL_Test" + shape_path.stem + "-Honeycomb-" + LoadingCase + f"_St{str(Strut_Thickness)[2:]}",
    material_bins=2,
    material_mass_density_kg_m3=1550.0, 
)
edge_lengths = honeycomb_fields["edge_curve_lengths_xyz"]          # [E]
total_all = honeycomb_fields["total_curve_length"]                 # scalar tensor
total_vd = honeycomb_fields["total_voronoi_curve_length"]  
limit =300        # scalar tensor
stress = verification.embedded_fem.fe.stress_vm_ip_active.detach().cpu().numpy().max(axis=1)
print("Above limit:", np.count_nonzero(stress > limit))
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))

show_surface_density(honeycomb_fields, density_key="rho")
grid = show_fem_distributions(
    verification,
    case_name="HoneyComb",
    stress_limit=300.0,
    bins=5,
    show_edges=True,
    clim_stress=(0, 300),       # MPa
    #clim_displacement=(0, 0.5),  # mm
)

In [ ]:
from Training.MainTrain import (
    _field_to_numpy,
    load_optimized_shell_function,
    evaluate_optimized_shell_function,
    visualize_optimized_shell_fields,
)

Path1 = "/home/arash/HVD_SeedsBase/Case_Studies/Wing_NACA2412_50_Sth4_ReaLCase/optimized_shell_function.pt"
path2 = "/home/arash/HVD_SeedsBase/Case_Studies/PlanarPlannar_Seeds-70_Sth033/BestFeasibleCheckpoint/optimized_shell_function.pt"

Mpath =Path1
if not os.path.exists(Mpath):
    raise FileNotFoundError(f"Optimized shell function file not found: {Mpath}")
opt_fn = load_optimized_shell_function(Mpath, device="cpu")
fields = evaluate_optimized_shell_function(opt_fn)
fields["rho"] = (fields["rho"]>0.5).to(torch.float64)
optimized_metrics = verification.run_fields(
    fields=fields,
    case_name=shape_path.stem + "-OptimizedModel_Bend_St0174_F5_V80_St025",
    material_bins=2,
    material_mass_density_kg_m3=1550.0, 
)

total_all = fields["total_curve_length"]                 # scalar tensor
total_vd = fields["total_voronoi_curve_length"]
stress = verification.embedded_fem.fe.stress_vm_ip_active.detach().cpu().numpy().max(axis=1)
print("Above limit:", np.count_nonzero(stress > limit)) 
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))
show_surface_density(fields, density_key="rho")
grid = show_fem_distributions(
    verification,
    case_name="Optimized",
    stress_limit=300.0,
    bins=5,
    show_edges=True,
    clim_stress=(0, 300),       # MPa
    #clim_displacement=(0, 0.5),  # mm
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs